# All in one Report Consolidation-Pipeline

### Importing required libraries

In [1]:
import pandas as pd
import glob
import datetime
import xlsxwriter
import os

### Starting time

In [2]:
start_time = datetime.datetime.now().strftime("%d-%m-%y-%H%M%S")
start_time

'11-06-26-161757'

## Report source

In [3]:
Source = r"D:\Learning\AR\Outstanding"

In [4]:
files = glob.glob(r""+ Source + "\\*.xlsx")
files

['D:\\Learning\\AR\\Outstanding\\Outstanding_Report_Dummy.xlsx']

In [5]:
files

['D:\\Learning\\AR\\Outstanding\\Outstanding_Report_Dummy.xlsx']

### Consolidating Reports and saving it as a DataFrame

In [6]:
def consolidate_reports(files, header_row=0, skip_sheets=[]):
    all_df = []
    
    for file in files:
        file_i = pd.ExcelFile(file)
        sheet_names = file_i.sheet_names
        sheet_count = len(sheet_names)
        
        data_sheets = [i for i in range(sheet_count) if i not in skip_sheets]
        
        if not data_sheets:
            continue
        
        if len(data_sheets) > 1:
            lst_of_df = []
            headers = None
            
            for i, sheet in enumerate(data_sheets):
                df = pd.read_excel(file, sheet_name=sheet, header=header_row if i == 0 else None)
                if i == 0:
                    headers = df.columns
                else:
                    df.columns = headers[:len(df.columns)]
                lst_of_df.append(df)
            
            df = pd.concat(lst_of_df, ignore_index=True)
        else:
            df = pd.read_excel(file, sheet_name=data_sheets[0], header=header_row)
        all_df.append(df)
    
    if not all_df:
        return pd.DataFrame()
    return pd.concat(all_df, ignore_index=True)
df = consolidate_reports(files, header_row=0, skip_sheets=[])

In [7]:
df.head(1)

,Business Unit,Customer Number,Customer Name,Customer Category,Bill Number,Bill Date,Type,Patient Name,Bill Amount,Settlement,Outstanding
0,AHEL Bhubaneswar BU,C-2035,MEDI ASSIST-GO DIGIT GENERAL INSURANCE LTD,Corporate,AH-BBS-IP-1000,12-Mar-2026,BHU IP Invoice,Jimmy McGill,63599,18340,45259


### Standard Unit Name Mapping

In [8]:
def std_unit_assign(df):
    if df["Business Unit"] == "AHEL Bhubaneswar BU":
        return "Bhubaneswar"
    elif df["Business Unit"] == "AHEL Kochi BU":
        return "Kochi"
    elif df["Business Unit"] == "AHEL Delhi BU":
        return "Delhi"
    elif df["Business Unit"] == "AHEL Ahmedabad BU":
        return "Ahmedabad"
    elif df["Business Unit"] == "AHEL Bangalore BU":
        return "Bangalore"
    elif df["Business Unit"] == "AHEL Pune BU":
        return "Pune"
    elif df["Business Unit"] == "AHEL Nagpur BU":
        return "Nagpur"
    elif df["Business Unit"] == "AHEL Hyderabad BU":
        return "Hyderabad"
    elif df["Business Unit"] == "AHEL Mumbai BU":
        return "Mumbai"
    elif df["Business Unit"] == "AHEL Chandigarh BU":
        return "Chandigarh"
    elif df["Business Unit"] == "AHEL Mysore BU":
        return "Mysore"
    elif df["Business Unit"] == "AHEL Kolkata BU":
        return "Kolkata"
    elif df["Business Unit"] == "AHEL Jaipur BU":
        return "Jaipur"
    elif df["Business Unit"] == "AHEL Chennai BU":
        return "Chennai"
    elif df["Business Unit"] == "AHEL Lucknow BU":
        return "Lucknow"
    else:
        return "New Unit"

In [9]:
df["Std Unit"] = df.apply(std_unit_assign,axis=1)

In [10]:
df.head(1)

,Business Unit,Customer Number,Customer Name,Customer Category,Bill Number,Bill Date,Type,Patient Name,Bill Amount,Settlement,Outstanding,Std Unit
0,AHEL Bhubaneswar BU,C-2035,MEDI ASSIST-GO DIGIT GENERAL INSURANCE LTD,Corporate,AH-BBS-IP-1000,12-Mar-2026,BHU IP Invoice,Jimmy McGill,63599,18340,45259,Bhubaneswar


### Standard Customer Name and Category Mapping

In [11]:
customer_master = pd.read_excel(r"D:\Learning\AR\Customer_Master\Payer_Customer_Master.xlsx")

In [12]:
customer_master.head(1)

,Payer_Name_Raw,Std_Payer_Name,Std_Payer_Category
0,MEDI ASSIST-GO DIGIT GENERAL INSURANCE LTD,Medi Assist Insurance TPA Pvt Ltd,TPA/Insurance


### Merging Customer Master df

In [13]:
df = df.merge(customer_master , left_on="Customer Name" , right_on="Payer_Name_Raw" , how = "left")

In [14]:
df.head(1)

,Business Unit,Customer Number,Customer Name,Customer Category,Bill Number,Bill Date,Type,Patient Name,Bill Amount,Settlement,Outstanding,Std Unit,Payer_Name_Raw,Std_Payer_Name,Std_Payer_Category
0,AHEL Bhubaneswar BU,C-2035,MEDI ASSIST-GO DIGIT GENERAL INSURANCE LTD,Corporate,AH-BBS-IP-1000,12-Mar-2026,BHU IP Invoice,Jimmy McGill,63599,18340,45259,Bhubaneswar,MEDI ASSIST-GO DIGIT GENERAL INSURANCE LTD,Medi Assist Insurance TPA Pvt Ltd,TPA/Insurance


### Converting Str formatted date into datetime format

In [15]:
df["Bill Date"]  = pd.to_datetime(df["Bill Date"],format='%d-%b-%Y')

In [16]:
df.head(1)

,Business Unit,Customer Number,Customer Name,Customer Category,Bill Number,Bill Date,Type,Patient Name,Bill Amount,Settlement,Outstanding,Std Unit,Payer_Name_Raw,Std_Payer_Name,Std_Payer_Category
0,AHEL Bhubaneswar BU,C-2035,MEDI ASSIST-GO DIGIT GENERAL INSURANCE LTD,Corporate,AH-BBS-IP-1000,2026-03-12,BHU IP Invoice,Jimmy McGill,63599,18340,45259,Bhubaneswar,MEDI ASSIST-GO DIGIT GENERAL INSURANCE LTD,Medi Assist Insurance TPA Pvt Ltd,TPA/Insurance


### Assigning Financial Year to rows

In [17]:
def FY_assign(consolidated):
    month = consolidated["Bill Date"].month
    year = consolidated["Bill Date"].year
    
    if consolidated["Bill Date"] < pd.Timestamp("2021-04-01"):
        return "Prior"
    
    elif month > 4:
        return f"FY {str(year)[-2:]}-{str(year+1)[-2:]}"
    
    else:
        return f"FY {str(year-1)[-2:]}-{str(year)[-2:]}"

In [18]:
df["Financial Year"] = df.apply(FY_assign,axis=1)

### Creating "Month Year" column

In [19]:
df["Month_Year"] = df["Bill Date"].dt.strftime("%b-%y")

In [20]:
df.head(1)

,Business Unit,Customer Number,Customer Name,Customer Category,Bill Number,Bill Date,Type,Patient Name,Bill Amount,Settlement,Outstanding,Std Unit,Payer_Name_Raw,Std_Payer_Name,Std_Payer_Category,Financial Year,Month_Year
0,AHEL Bhubaneswar BU,C-2035,MEDI ASSIST-GO DIGIT GENERAL INSURANCE LTD,Corporate,AH-BBS-IP-1000,2026-03-12,BHU IP Invoice,Jimmy McGill,63599,18340,45259,Bhubaneswar,MEDI ASSIST-GO DIGIT GENERAL INSURANCE LTD,Medi Assist Insurance TPA Pvt Ltd,TPA/Insurance,FY 25-26,Mar-26


### Ageing Bucket

##### Current Date

In [21]:
today = pd.to_datetime(datetime.datetime.now().date(), format='%d:%b:%y:%H:%M:%S')
today

Timestamp('2026-06-11 00:00:00')

##### To get Last date of data ( for closing month purpose )

In [22]:
max_bill_date = pd.to_datetime(df["Bill Date"],format='%d:%b:%y:%H:%M:%S').max()
max_bill_date

Timestamp('2026-05-21 00:00:00')

##### Days Difference

In [23]:
df["Ageing"] = (today - df["Bill Date"]).dt.days

##### Ageing Bucket Assign

In [24]:
def Ageing_Bucket(consolidated):
    if consolidated["Ageing"] <= 30:
        return "1) 0D-30D"
    elif consolidated["Ageing"] <= 60:
        return "2) 31D-60D"
    elif consolidated["Ageing"] <= 90:
        return "3) 61D-90D"
    elif consolidated["Ageing"] <= 120:
        return "4) 91D-120D"
    elif consolidated["Ageing"] <= 180:
        return "5) 121D-180D"
    elif consolidated["Ageing"] <= 365:
        return "6) 181D-1Y"
    elif consolidated["Ageing"] <= 730:
        return "7) 1Y-2Y"
    elif consolidated["Ageing"] <= 1095:
        return "8) 2Y-3Y"
    elif consolidated["Ageing"] > 1095:
        return "9) 3Y+"
    else:
        return "N/A"

In [25]:
df["Ageing Bucket"] = df.apply(Ageing_Bucket,axis=1)

### Claim Submission

In [26]:
claim_submission = pd.read_excel(r"D:\Learning\AR\Claim Submission\Claim_Submission_Report_Dummy.xlsx")[["Claim Bill Number","Claim ID"]]

In [27]:
claim_submission.head(1)

,Claim Bill Number,Claim ID
0,AH-BBS-IP-1000,CLM-BBS-001001


### Merging Claim Submission df

In [28]:
df = df.merge(claim_submission,left_on="Bill Number" , right_on = "Claim Bill Number" , how = "left")

In [29]:
df.head(1)

,Business Unit,Customer Number,Customer Name,Customer Category,Bill Number,Bill Date,Type,Patient Name,Bill Amount,Settlement,...,Std Unit,Payer_Name_Raw,Std_Payer_Name,Std_Payer_Category,Financial Year,Month_Year,Ageing,Ageing Bucket,Claim Bill Number,Claim ID
0,AHEL Bhubaneswar BU,C-2035,MEDI ASSIST-GO DIGIT GENERAL INSURANCE LTD,Corporate,AH-BBS-IP-1000,2026-03-12,BHU IP Invoice,Jimmy McGill,63599,18340,...,Bhubaneswar,MEDI ASSIST-GO DIGIT GENERAL INSURANCE LTD,Medi Assist Insurance TPA Pvt Ltd,TPA/Insurance,FY 25-26,Mar-26,91,4) 91D-120D,AH-BBS-IP-1000,CLM-BBS-001001


### Identifying Self Paid Cases

If Bills are paid partially by payer , that's "Partial Paid". If paid partially by Patient that's not Partial Paid. It's actually the credit bill's value is getting reduced. So , the bill could be fall under "Fully Unpaid" category. That's why below script is being used.

##### Accounting Report

In [30]:
accounting = pd.read_excel(r"D:\Learning\AR\Accounting\Accounting_Report_Dummy.xlsx")

##### Filtering only Self Paid cases 

In [31]:
self_paid_accounting = accounting[accounting["Customer Category"].isin(["Self Pay"])]

In [32]:
self_paid_accounting.head(1)

,l,Payment Receipt Date,Receipt Method,Receipt Cust Code,Receipt Customer Name,Customer Category,Receipt Number,Actual Receipt Amount,Bill Cust Code,Bill Customer Name,Accounting Bill Number,Bill Date,Bill Amount,GL Accounting Date,Accounted Amount
1,AHEL Kochi BU,25-May-2026,ICICI-CA-000205032145,C-2531,SELF PAY,Self Pay,NEFT66691283914,30846,C-2589,SELF PAY,AH-COK-OP-2000,13-Mar-2026,56985,28-May-2026,30846


##### Selecting only columns that needed in the process

In [33]:
accounting = self_paid_accounting[["Accounting Bill Number","Accounted Amount"]].rename(columns={"Accounted Amount":"Self Paid Amount"})

##### Merging accounting df

In [34]:
df = df.merge(accounting , left_on="Bill Number" , right_on= "Accounting Bill Number" , how= "left")

##### Difference between "Bill Amount" vs "Outstanding"

In [35]:
df["BillAmount_vs_Outstanding"] = df["Bill Amount"] - df["Outstanding"]

##### Difference between "Self Paid AMount" vs "Bill Amount"

In [36]:
df["SelfPaid_vs_BillAmount"] = df["Bill Amount"] - df["Self Paid Amount"]

##### Difference between "SelfPaid_vs_BillAmount" vs "Outstanding"

In [37]:
df["Diff_vs_Outstanding"] = df["SelfPaid_vs_BillAmount"] - df["Outstanding"]

In [38]:
df.head(1)

,Business Unit,Customer Number,Customer Name,Customer Category,Bill Number,Bill Date,Type,Patient Name,Bill Amount,Settlement,...,Month_Year,Ageing,Ageing Bucket,Claim Bill Number,Claim ID,Accounting Bill Number,Self Paid Amount,BillAmount_vs_Outstanding,SelfPaid_vs_BillAmount,Diff_vs_Outstanding
0,AHEL Bhubaneswar BU,C-2035,MEDI ASSIST-GO DIGIT GENERAL INSURANCE LTD,Corporate,AH-BBS-IP-1000,2026-03-12,BHU IP Invoice,Jimmy McGill,63599,18340,...,Mar-26,91,4) 91D-120D,AH-BBS-IP-1000,CLM-BBS-001001,NaN,NaN,18340,NaN,NaN


#### Assigning Invoice Status

In [39]:
def partial_tags(df):
    if df["Type"] == "Payments":
        return "Unaccounted Payments"
    elif df["Outstanding"] < 0:
        return "Non-AR"
    elif df["Diff_vs_Outstanding"] == 0:  # Check this FIRST before Partially Paid
        return "Fully Unpaid"
    elif (df["Outstanding"] / df["Bill Amount"]) < 0.995:
        return "Partially Paid"
    elif (df["Outstanding"] / df["Bill Amount"]) >= 0.995:
        return "Fully Unpaid"
    else:
        return "Other"

In [40]:
df["Invoice Status"] = df.apply(partial_tags,axis=1)

In [41]:
df.head(1)

,Business Unit,Customer Number,Customer Name,Customer Category,Bill Number,Bill Date,Type,Patient Name,Bill Amount,Settlement,...,Ageing,Ageing Bucket,Claim Bill Number,Claim ID,Accounting Bill Number,Self Paid Amount,BillAmount_vs_Outstanding,SelfPaid_vs_BillAmount,Diff_vs_Outstanding,Invoice Status
0,AHEL Bhubaneswar BU,C-2035,MEDI ASSIST-GO DIGIT GENERAL INSURANCE LTD,Corporate,AH-BBS-IP-1000,2026-03-12,BHU IP Invoice,Jimmy McGill,63599,18340,...,91,4) 91D-120D,AH-BBS-IP-1000,CLM-BBS-001001,NaN,NaN,18340,NaN,NaN,Partially Paid


### Removing unwanted columns

In [42]:
cols_to_drop = ["Claim Bill Number","Accounting Bill Number","Payer_Name_Raw"]

In [43]:
df = df.drop(columns=cols_to_drop)

### Reordering Columns

In [44]:
cols_order = [
            "Business Unit",
            "Std Unit",
            "Customer Number",
            "Customer Name",
            "Customer Category",
            "Bill Number",
            "Bill Date",
            "Type",
            "Patient Name",
            "Bill Amount",
            "Settlement",
            "Outstanding",
            "Std_Payer_Name",
            "Std_Payer_Category",
            "Financial Year",
            "Month_Year",
            "Ageing",
            "Ageing Bucket",
            "Claim ID",
            "BillAmount_vs_Outstanding",
            "Self Paid Amount",
            "SelfPaid_vs_BillAmount",
            "Diff_vs_Outstanding",
            "Invoice Status",
            ]

In [45]:
df = df.reindex(columns=cols_order)

### Writing the final DataFrame as an Excel file

In [46]:
with pd.ExcelWriter(r"D:\Learning\AR\Consolidated Outstanding with working-"+start_time+".xlsx",engine="xlsxwriter") as writer:
    df.to_excel(writer,sheet_name="Consolidated",index=False)

##### End Time

In [47]:
end_time = datetime.datetime.now().strftime("%d-%m-%y-%H%M%S")
end_time

'11-06-26-161758'

### To calculate the total time elapsed for this process

In [48]:
timedelta = pd.to_datetime(end_time, format = "%d-%m-%y-%H%M%S",errors="coerce") - pd.to_datetime(start_time, format = "%d-%m-%y-%H%M%S",errors="coerce")

In [49]:
timedelta = str(timedelta).split("days")[-1].strip()
print("Total Time Elapsed: " + timedelta)

Total Time Elapsed: 00:00:01
